# Estudo de caso 4.1 - Sistema de recomendação de filmes

#### Observação: Se em algum momento este notebook for encerrado, você terá que executar novamente todas as células ao reabri-lo.

#### Observação: É possível que haja diferentes resultados numéricos se o notebook for executado em diferentes ocasiões. Isto é normal. Simplesmente entregue os resultados obtidos.

## PYTHON AVANÇADO

Como esta é a versão avançada, não incluímos parte do código nela. Se você ficar "preso" em alguma parte do estudo de caso, não hesite em consultar a [versão para iniciantes](https://drive.google.com/file/d/1ie1GfaDP5fjhpzE68ysxek_xTEubvn4k/view?usp=sharing) do estudo para obter ajuda.

## Informações de contato

In [ ]:
# SEU NOME                = ...
# SEU USUÁRIO MITX PRO    = ...
# SEU E-MAIL MITX PRO     = ...

## Configuração

Execute (Run) estas células para instalar os pacotes necessários para a realização do estudo de caso. Tenha paciência, pois isso poderá levar alguns minutos.

In [ ]:
!pip install --upgrade pip
!pip install surprise==0.1
print('Bibliotecas instaladas com sucesso!')

Se nenhum texto em vermelho indicando erro aparecer, isso significa que a instalação foi bem-sucedida. Os textos em amarelo são avisos, mas não erros.

<h1>Atenção:</h1>

Agora, reinicie o ambiente de execução. Para isso, vá para:

> Ambiente de execução > _Reiniciar ambiente de execução_ 

na parte superior da tela. Isto irá garantir que suas alterações foram feitas com sucesso.

## Importar

Importe as bibliotecas necessárias para o desenvolvimento do estudo de caso.

In [ ]:
import pandas as pd
import matplotlib
from surprise import Dataset, SVD, NormalPredictor, BaselineOnly, KNNBasic, NMF
from surprise.model_selection import cross_validate, KFold
%matplotlib inline
print('Bibliotecas importadas com sucesso!')

## Dados

Use a função [`Dataset.load_builtin`](http://surprise.readthedocs.io/en/stable/dataset.html#surprise.dataset.Dataset.load_builtin) para carregar o banco de dados `ml-100k` de *MovieLens*.

In [ ]:
# Escreva aqui seu código para carregar os dados...

Visto que também queremos ter uma ideia da aparência dos dados, crie um histograma de todas as pontuações existentes no banco de dados.

In [ ]:
# Escreva aqui seu código para criar um histograma das pontuações...

<h1>PERGUNTA 1: ANÁLISE DE DADOS</h1>

**Descreva o banco de dados. Quantas pontuações (ratings) existem no banco de dados? Como você descreveria a distribuição das pontuações? Há algo mais que devemos observar?**

Certifique-se de que o histograma possa ser visualizado no notebook inclusive após ter sido convertido para o formato .pdf.

*Escreva aqui sua resposta...*

## Modelo 1: aleatório

In [ ]:
# Crie um modelo usando a classe NormalPredictor()

In [ ]:
# Treine o modelo com os dados usando a validação cruzada com k=5 
# iterações, medindo o RMSE 
# Observe a função cross_validate que importamos no início 
# http://surprise.readthedocs.io/en/stable/model_selection.html#surprise.model_selection.validation.cross_validate

## Modelo 2: filtragem colaborativa baseada no usuário

In [ ]:
# Crie um modelo usando a classe KNNBasic()
# Veja o paramêtro sim_options para determinar a similaridade usuario/item calculada do modelo
# http://surprise.readthedocs.io/en/stable/prediction_algorithms.html#similarity-measures-configuration

In [ ]:
# Treine o modelo usando o mesmo esquema de validação cruzada
# usado anteriormente

## Modelo 3: filtragem colaborativa baseada no item

In [ ]:
# Crie um modelo usando a classe KNNBasic()
# Tenha certeza de mudar o parâmetro sim_options do código acima

In [ ]:
# Treine o modelo usando o mesmo esquema de validação cruzada
# usado anteriormente

<h1>PERGUNTA 2: MODELOS DE FILTRAGEM COLABORATIVA</h1>

**Compare os resultados dos modelos de filtragem colaborativa baseada em usuários e em itens. Que diferenças você encontra entre os dois? Que diferenças você encontra entre esses modelos e o modelo aleatório? Consegue explicar o que pode ter causado essas diferenças nos resultados?**

*Escreva aqui sua resposta...*

## Modelo 4: fatoração de matriz

In [ ]:
# Crie um modelo usando a classe SVD() 

In [ ]:
# Treine o modelo usando o mesmo esquema de validação cruzada
# usado anteriormente

<h1>PERGUNTA 3: MODELO DE FATORAÇÃO DE MATRIZ</h1>

**O modelo de fatoração de matriz é diferente dos modelos de filtragem colaborativa. Descreva resumidamente em que consistem essas diferenças. Além disso, compare o RMSE novamente em relação ao resto dos modelos. É melhor ou pior? Poderia explicar por que melhora/piora?**

*Escreva aqui sua resposta...*

## Precisão e exatidão @`k` (*precision and recall @k*)

Queremos calcular a precisão e a exatidão para 2 valores de k: 5 e 10. Incluímos algumas linhas de código que lhe ajudarão nisso.

Primeiro, definimos uma função que usa algumas previsões, um valor de k e um parâmetro de limiar. Este código foi adaptado da seguinte [fonte](http://surprise.readthedocs.io/en/stable/FAQ.html?highlight=precision#how-to-compute-precision-k-and-recall-k).

**Certifique-se de executar esta célula**

In [ ]:
def precision_recall_at_k(predictions, k=10, threshold=3.5):
    '''Retorna a precisão e exatidão @k para cada usuário'''

    # Primeiro, associe as previsões a cada usuário
    user_est_true = dict()
    for uid, _, true_r, est, _ in predictions:
        current = user_est_true.get(uid, list())
        current.append((est, true_r))
        user_est_true[uid] = current

    precisions = dict()
    recalls = dict()
    for uid, user_ratings in user_est_true.items():

        # Ordene as pontuações dos usuários por seu valor estimado
        user_ratings.sort(key=lambda x: x[0], reverse=True)

        # Número de itens relevantes
        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)

        # Número de itens recomendados no top k
        n_rec_k = sum((est >= threshold) for (est, _) in user_ratings[:k])

        # Número de itens relevantes e recomendados no top k
        n_rel_and_rec_k = sum(((true_r >= threshold) and (est >= threshold))
                              for (est, true_r) in user_ratings[:k])

        # Precisão@k: proporção de itens recomendados que são relevantes
        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 1

        # Exatidão@K: proporção de itens relevantes que são recomendados
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 1

    return precisions, recalls

print('\n\nFunção criada com sucesso!')

Em seguida, calculamos a precisão e a exatidão @`k`= 5 e 10. Usamos a validação cruzada com 5 iterações novamente para calcular a média dos resultados de todo o banco de dados.

Seja paciente, pois a execução pode demorar um pouco.

<h1>PERGUNTA 4: PRECISÃO/EXATIDÃO</h1>

**Calcule a precisão e exatidão, para cada um dos 4 modelos, com `k` = 5 e 10. Ou seja, 2 x 2 x 4 = 16 valores numéricos. Percebe algo diferente nesses valores? Há algo diferente dos valores de RMSE calculados anteriormente?**

Para esta pergunta, você precisará escrever um código:

In [ ]:
# Use a função precision_recall_at_k() que acaba de ser definida 
# para calcular os 16 valores numéricos

# Consulte a função test() para obter as previsões que servem de 
# entrada para a função precision_recall_at_k()
# http://surprise.readthedocs.io/en/stable/algobase.html#surprise.prediction_algorithms.algo_base.AlgoBase.test

*Escreva aqui sua resposta...*

##  Top-`n` Previsões

Finalmente, queremos ver como são as recomendações e as estimativas de pontuações dos usuários.

Novamente, definimos uma função auxiliar.

In [ ]:
def get_top_n(predictions, n=5):
    '''Retorna as  top-N recomendações para cada usuário de um conjunto de previsões.

    Argumentos:
        predictions(lista de objetos de previsão): lista das previsões, 
            na forma em que são obtidas do método "test" de um algoritmo
        n(int): número de recomendações a serem exibidas para cada usuário. 
            Por padrão é 10.

    Saídas:
    Um dicionário onde as keys são as IDs dos usuários e os valores são
    uma lista de tuplas:
        [(item id, estimativa da pontuação), ...] de tamanho n.
    '''

    # Primeiro, associe as previsões a cada usuário.
    top_n = dict()
    for uid, iid, true_r, est, _ in predictions:
        current = top_n.get(uid, [])
        current.append((iid, est))
        top_n[uid] = current

    # Depois, ordene as previsões para cada usuário e obtenha a 
    # n previsões mais elevadas
    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]

    return top_n

print('Função criada com sucesso!')

Por fim, executamos esta função em cada um dos modelos, primeiro treinando em **todos** os dados disponíveis e, em seguida, prevendo os dados ausentes. Usamos `n` = 5, mas você pode escolher qualquer valor razoável de n.

Isso pode levar algum tempo de computação, por isso tenha paciência.

Dica: Use [`Dataset.build_full_trainset`](http://surprise.readthedocs.io/en/stable/dataset.html#surprise.dataset.DatasetAutoFolds.build_full_trainset) para obter o conjunto de treinamento do banco de dados. Depois, execute [`Trainset.build_anti_testset`](http://surprise.readthedocs.io/en/stable/trainset.html#surprise.Trainset.build_anti_testset) para obter o conjunto de teste. Por último, use `fit` no conjunto de treinamento, `test` no conjunto de teste e passe o resultado para a função `get_top_n()`.

<h1>PERGUNTA 5: TOP-N PREVISÕES</h1>

**As top n previsões obtidas fazem sentido? Qual é o valor das pontuações (1-5) dessas previsões? Como você poderia usar essas previsões na vida real se estivesse tentando construir um sistema de recomendação genérico para uma empresa?**

Para esta pergunta, você precisará escrever um código:

In [ ]:
# Use a função get_top_n() e as dicas para obter as top-n recomendações
# para um determinado usuário, para um valor razoável de n

*Escreva aqui sua resposta...*

<hr>

Bom trabalho! Verifique a seção **Entrega** do manual de instruções para concluir e entregar este estudo corretamente.